# S3 — Pandas 轉換：groupby / merge / pivot

> **時間**：2 小時  
> **資料**：`orders_clean.csv`（S2 產出）+ `customers.csv` + `products.csv`  
> **學完能幹嘛**：用 SQL 的思維在 Pandas 中做三表 join、分組統計、樞紐分析

---

## 上節回顧 + 本節為什麼重要

S2 我們把髒訂單清乾淨了，存成 `orders_clean.csv`。但單看一張訂單表只看得到數字，看不到**商業洞察**，例如：

- 哪個**地區**的客人買最多？ → 需要把 orders 和 customers join
- 哪個**商品類別**貢獻最多營收？ → 需要把 orders 和 products join
- 每個地區的 Top 3 熱銷商品？ → 需要 groupby + 排序

> **DE/DA 視角**：如果你會 SQL，本節你會很爽；如果不會，本節學完你就等於會了 80% 的 SQL。

**三大工具**：
1. **`merge`** = SQL 的 JOIN（合併多表）
2. **`groupby`** = SQL 的 GROUP BY（分組統計）
3. **`pivot_table`** = Excel 的樞紐分析表（二維交叉統計）


In [ ]:
import pandas as pd
import numpy as np

DATA = '../datasets/ecommerce'
orders    = pd.read_csv(f'{DATA}/orders_clean.csv', parse_dates=['order_date'])
customers = pd.read_csv(f'{DATA}/customers.csv')
products  = pd.read_csv(f'{DATA}/products.csv')

print('orders   :', orders.shape,    list(orders.columns))
print('customers:', customers.shape, list(customers.columns))
print('products :', products.shape,  list(products.columns))

---
## 核心觀念 1／3 — 條件篩選（複習 + 加強）

S2 學過 `df.loc[mask]`，本節加強兩招：`isin` 和 `between`。


In [ ]:
# 1) 找出金額在 1000~5000 之間的訂單
mid = orders[orders['amount'].between(1000, 5000)]
print('中間金額訂單:', len(mid))

# 2) 找出指定顧客的訂單（用 isin 一次篩多個值）
vip_ids = [2001, 2005, 2010]
vip_orders = orders[orders['customer_id'].isin(vip_ids)]
print('VIP 訂單:', len(vip_orders))

# 3) 複合條件：高金額 + 特定顧客
hit = orders[(orders['amount'] > 3000) & (orders['customer_id'].isin(vip_ids))]
print('複合條件命中:', len(hit))

**口訣**：`between` 找區間、`isin` 找清單、`&/|` 組合條件（記得加括號）。


---
## 核心觀念 2／3 — `groupby`：分組 + 聚合

**SQL 對照**：
```sql
SELECT customer_id, SUM(amount) 
FROM orders 
GROUP BY customer_id;
```
↓ 轉成 Pandas ↓
```python
orders.groupby('customer_id')['amount'].sum()
```


In [ ]:
# 1) 最簡單：每個顧客的總消費
per_customer = orders.groupby('customer_id')['amount'].sum()
print(per_customer.head())
print('型別:', type(per_customer).__name__)  # Series

In [ ]:
# 2) 多種聚合一次做：count, sum, mean
summary = orders.groupby('customer_id')['amount'].agg(['count', 'sum', 'mean'])
summary.columns = ['訂單數', '總金額', '平均金額']
print(summary.head())

# 3) 排序找 Top 5
print('\n💰 總金額 Top 5 顧客:')
print(summary.sort_values('總金額', ascending=False).head())

In [ ]:
# 4) 多欄聚合：不同欄用不同函式
multi = orders.groupby('customer_id').agg(
    訂單數=('order_id', 'count'),
    總金額=('amount', 'sum'),
    平均金額=('amount', 'mean'),
    最大單筆=('amount', 'max'),
).reset_index()
multi.head()

**口訣**：`groupby(分組欄).agg(...)` → 一行程式替代 SQL 的整段 GROUP BY。


> ### 💡 知識補給站 — groupby 背後的 Hash Table
> 
> `groupby('customer_id')` 底層做了什麼？Pandas 建了一個 **hash table**（雜湊表），把每個 `customer_id` 映射到它出現的**行號清單**，然後對每組套用聚合函式。
> 
> - 這跟 SQL 的 `GROUP BY` 概念一模一樣 — 資料庫也是用 **hash aggregate** 或 sort aggregate
> - Hash table 的查找效率是 **O(1)**，所以 groupby 整體效能是 **O(n)** — 100 萬筆訂單也能在毫秒內完成
> - 效能瓶頸不在分組數量，而在每組的聚合運算複雜度
> 
> 這也是為什麼 Pandas 的 `groupby` 跟用 `for` 迴圈自己分組比起來快非常多 — 底層是 C 語言實作的 hash table，不是 Python 層級的字典操作。
> 
> *延伸關鍵字：hash table, hash aggregate, O(n) complexity, data structure*

---
## 核心觀念 3／3 — `merge`：合併多表（= SQL JOIN）

四種 join 一張圖記熟：

| `how=` | 意義 | 什麼時候用 |
|---|---|---|
| `'inner'` | 兩邊都有才保留 | 最安全、最常用 |
| `'left'`  | 保留左表全部 | 以主表為準 |
| `'right'` | 保留右表全部 | 少用 |
| `'outer'` | 全部都保留 | 找遺漏用 |


In [ ]:
# 1) 兩表 join：orders + customers（看每筆訂單屬於哪個地區）
oc = orders.merge(customers, on='customer_id', how='left')
print('合併後形狀:', oc.shape)
print('新增的欄位:', [c for c in oc.columns if c not in orders.columns])
oc.head()

In [ ]:
# 2) 三表 join：再加上 products
#    注意：orders 裡是 product_id，products 裡也是 product_id → on='product_id'
full = (
    orders
    .merge(customers, on='customer_id', how='left')
    .merge(products,  on='product_id',  how='left')
)
print('三表合併後形狀:', full.shape)
print('欄位:', list(full.columns))
full.head()

**口訣**：`merge(另一張表, on='共同欄', how='left/inner')` — 從左表的角度去黏上右表的資訊。


> ### 💡 知識補給站 — merge 的隱藏陷阱：validate 參數
> 
> `merge` 預設**不檢查 key 的唯一性**。如果 `products.csv` 裡同一個 `product_id` 出現兩次，merge 後列數會**暴增**（笛卡爾積），而且**不會報錯** — 這是真實工作中最常見的 silent data quality bug。
> 
> 解法：加上 `validate` 參數，key 不符預期會直接噴 `MergeError`：
> ```python
> orders.merge(products, on='product_id', how='left', validate='many_to_one')
> ```
> 
> | validate 值 | 意義 |
> |---|---|
> | `'one_to_one'` | 兩邊的 key 都必須唯一 |
> | `'many_to_one'` | 右表的 key 必須唯一（最常用） |
> | `'one_to_many'` | 左表的 key 必須唯一 |
> 
> 類比：結婚登記處會檢查「雙方都是未婚」— 不檢查就會出問題。
> 
> **實務建議**：每次 merge 完第一件事就是 `print(df.shape)` 確認列數是否合理。
> 
> *延伸關鍵字：merge validate, Cartesian product, data quality, join explosion*

---
## 實務 Case — 電商銷售洞察分析

**情境**：老闆下午開會要你回答三個問題：
1. 🏆 各**地區**的總營收排名？
2. 🔥 各地區的 **Top 3 熱銷商品**是什麼？
3. 💎 **VIP 等級**的營收貢獻佔比？


In [ ]:
# 先把三表合併起來當分析基底
df = (
    orders
    .merge(customers, on='customer_id', how='left')
    .merge(products,  on='product_id',  how='left')
)
print('分析基底:', df.shape)
df[['order_id','customer_name','region','vip_level','product_name','category','amount']].head()

In [ ]:
# Q1: 各地區總營收排名
region_rev = df.groupby('region')['amount'].sum().sort_values(ascending=False)
print('🏆 地區營收排名:')
print(region_rev)

In [ ]:
# Q2: 各地區 Top 3 熱銷商品（by 銷售金額）
#     技巧：先 groupby 兩層 → sort → 每個地區取前 3
product_by_region = (
    df.groupby(['region', 'product_name'])['amount']
      .sum()
      .reset_index()
      .sort_values(['region', 'amount'], ascending=[True, False])
)

top3 = product_by_region.groupby('region').head(3)
print('🔥 各地區 Top 3 熱銷商品:')
print(top3)

In [ ]:
# Q3: VIP 等級貢獻佔比（用 pivot_table 或 groupby 都可以）
vip_rev = df.groupby('vip_level')['amount'].sum()
vip_pct = (vip_rev / vip_rev.sum() * 100).round(1)

vip_report = pd.DataFrame({'總金額': vip_rev, '佔比(%)': vip_pct})
vip_report = vip_report.sort_values('總金額', ascending=False)
print('💎 VIP 等級貢獻:')
print(vip_report)

In [ ]:
# Bonus: pivot_table — 交叉看「地區 × 品類」的營收矩陣
pivot = df.pivot_table(
    index='region',
    columns='category',
    values='amount',
    aggfunc='sum',
    fill_value=0,
)
print('📊 地區 × 品類 營收交叉表:')
print(pivot)

In [ ]:
# 存一份合併好的分析基底給 S4/S5/S6 用
df.to_csv('../datasets/ecommerce/orders_enriched.csv', index=False)
print('已存 orders_enriched.csv，供後續 session 使用')

---
## 課堂練習（40 min）

### 🟢 送分題
用 `orders` 算出：
1. 每個 `customer_id` 的訂單總數
2. 平均單筆訂單金額


In [ ]:
# TODO


### 🟡 核心題
用三表合併的 `df`，回答：
1. 哪個**商品類別**總營收最高？
2. **Gold** 等級的顧客一共下了多少筆訂單？金額多少？
3. 各**地區**的平均訂單金額是多少？


In [ ]:
# TODO


### 🔴 挑戰題
計算每位顧客的 **RFM 粗估**：
- **R**ecency：最近一次下單日期（用 `max(order_date)`）
- **F**requency：下單次數
- **M**onetary：總金額

最後用 M 排序，列出 Top 5 顧客（加上顧客名字）。

> 這是電商最經典的顧客分群指標，會做這題你就有 junior DA 實力了。


In [ ]:
# TODO


---
## 小結與速查表

| 想做什麼 | 怎麼寫 | SQL 對照 |
|---|---|---|
| 區間篩選 | `df[df['x'].between(a,b)]` | `WHERE x BETWEEN a AND b` |
| 清單篩選 | `df[df['x'].isin([...])]` | `WHERE x IN (...)` |
| 分組加總 | `df.groupby('k')['v'].sum()` | `GROUP BY k` |
| 多聚合 | `df.groupby('k').agg(...)` | 多 `SUM/COUNT/AVG` |
| Join 兩表 | `a.merge(b, on='k', how='left')` | `LEFT JOIN` |
| 樞紐分析 | `df.pivot_table(index, columns, values)` | (Excel 樞紐) |
| 每組 Top N | `sort + groupby.head(n)` | `ROW_NUMBER()` |

**下節預告 S4**：我們有 `order_date`，但還沒用它做時序分析。下節會教你用 1 行程式算出「月度營收」「7 日移動平均」「年月週趨勢」。
